# Feature Engineering

- Transform preprocessed data from `cookie`, `index`, `method`, `url`, `host`, and `label`, `payload` into numerical or categorical features that can be used for modeling.
- Focus on features that distinguish anomalous requests (anom) from normal requests (norm).
- Address missing values in contentLength (e.g., fill with 0 or add a binary flag).
- Convert categorical columns `method`, `label`, `host` into a format suitable for machine learning (e.g., one-hot encoding, label encoding).
- Extract features that capture HTTP request characteristics (e.g., SQL injection patterns, payload length, URL length).
- Save the transformed DataFrame with features as `csic_2010_featurised.csv` for modeling.

In [1]:
## 1. Load the Preprocessed Data

In [25]:
import os
import random
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

path = Path.cwd()
data_path = path.parent / "data"
input_path = data_path / "csic_2010_preprocessed.csv" 
output_path = data_path / "csic_2010_featurised.csv" 
# Load the DataFrame with keep_default_na=False to preserve empty strings
#df = pd.read_csv(input_path, keep_default_na=False)

In [26]:
# Load the Parquet file
%time
df = pd.read_parquet(input_path.with_suffix('.parquet'))

CPU times: user 19 μs, sys: 4 μs, total: 23 μs
Wall time: 628 μs


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


In [27]:
# Extract the `Payload` columns series
payload_series = df["payload"].fillna("").astype(str)

# Extract the `URL` columns series
url_series = df["url"].fillna("").astype(str)

# Extract the `cookie` columns series
cookie_series = df["cookie"].fillna("").astype(str)

## 2. Handle Missing Values

- Handle missing `contentLength` - fill with `0`
- Handle Empty `payload` - add binary flag to indicate whether re has payload

In [28]:
df['contentLength'] = df['contentLength'].fillna(0)

## 3. Analyse Column values to derive feature columns

In [29]:
print(f"cookie:        {df['cookie'].nunique()}")
print(f"index:         {df['index'].nunique()}")
print(f"method:        {df['method'].nunique()}")
print(f"url:           {df['url'].nunique()}")
print(f"host:          {df['host'].nunique()}")
print(f"label:         {df['label'].nunique()}")
print(f"payload:       {df['payload'].nunique()}")
print(f"contentLength: {df['contentLength'].nunique()}")

cookie:        61065
index:         36000
method:        3
url:           1643
host:          2
label:         2
payload:       12093
contentLength: 383


`index` column is non-unique and reused across requests so it is not useful in modeling. Drop the column.

In [8]:
df_featurised = df.drop(columns = 'index')

Sample few values of each columns for analysis

### Analyse `cookie` column values

In [10]:
# Random cookie samples
random_row_numbers = random.sample(range(0, 61066), 5)
for rn in random_row_numbers:
    print(f"row#{rn}: {df_featurised.iloc[rn]['cookie']}")

row#8198: JSESSIONID=2205F1840028B1374AABDCFF9B3368BB
row#51094: JSESSIONID=D55D916D32286C6970E15328AB3ED1A0
row#15574: JSESSIONID=40CB743ED8B770A898814D47FCF21EE6
row#14650: JSESSIONID=3CC65F29A19930EFAF372550816E6345
row#16753: JSESSIONID=45C9C3FE760FDC86895F43C9727CF1CE


In [11]:
# Compute the length of each cookie
cookie_lengths = df_featurised['cookie'].str.len()

print(f"Min len: {cookie_lengths.min()}")
print(f"Max len: {cookie_lengths.max()}")
print(f"mean len: {cookie_lengths.mean():.2f}")
#print(f"Unique lengths: {cookie_lengths.nunique()}")

# Count how many cookies match the pattern
cookie_pattern=r'^JSESSIONID=[A-Z0-9]+$'
valid_count = df_featurised['cookie'].str.match(cookie_pattern).sum()
print(f"Number of valid cookies: {valid_count}")
#print(f"Total cookies: {len(df_featurised)}")

Min len: 43
Max len: 43
mean len: 43.00
Number of valid cookies: 61065


All cookies have a length of `43` characters and a consistent format. This implies that the cookie column does not provide any variability that could distinguish between normal (norm) and anomalous (anom) requests. Drop the `cookie` Column from the Feature set

In [12]:
df_featurised = df.drop(columns = 'cookie')

### Analyse `Method` column values

In [13]:
df_featurised['method'].value_counts()

method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64

Indicates type of HTTP requests - majority are `GET` `43,088` followed by `POST` `17,580` and the `PUT` `397` which can influence anomaly detection  
e.g., `POST` and `PUT` requests are more likely to contain payloads or be anomalous  

Use one-hot encoding to transform method categories into numerical features.

In [14]:
# Initialize the encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["method"]

# Fit the encoder to the data
encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{encoder.categories_}")

# Generate column names for the encoded features
encoded_cols = list(encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{encoded_cols}")

Encoder categories:
	[array(['GET', 'POST', 'PUT'], dtype=object)]
Generated column names:
	['method_GET', 'method_POST', 'method_PUT']


The method column is transformed into three binary features - `method_GET`, `method_POST`, `method_PUT`. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and Concatenate the encoded features with the original DataFrame for modeling.

In [15]:
df_featurised[encoded_cols] = encoder.transform(df_featurised[category_col]).copy()

In [16]:
# Drop the original 'method' column as it is longer needed
df_featurised.drop(columns=["method"], inplace=True)

In [17]:
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          61065 non-null  int64  
 1   url            61065 non-null  object 
 2   host           61065 non-null  object 
 3   label          61065 non-null  object 
 4   payload        61065 non-null  object 
 5   contentLength  61065 non-null  float64
 6   method_GET     61065 non-null  float64
 7   method_POST    61065 non-null  float64
 8   method_PUT     61065 non-null  float64
dtypes: float64(4), int64(1), object(4)
memory usage: 4.2+ MB


### Analyse `host` column values

In [18]:
df_featurised['host'].value_counts()

host
localhost:8080    60668
localhost:9090      397
Name: count, dtype: int64

`host` is what type of feature category column ? give justificaiton for documentation

- Identifies the `server:port` for the request.
- Contains two host values `localhost:8080` and `localhost:9090`  
- Majority are `localhost:8080` `60,668` following by `localhost:9090` `397`
- Useful for filtering or grouping requests by host.
- `host` column is highly imbalanced, which may limit its predictive power.

In [20]:
# Initialize the encoder
host_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["host"]

# Fit the encoder to the data
host_encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{host_encoder.categories_}")

# Generate column names for the encoded features
host_encoded_cols = list(host_encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{host_encoded_cols}")

Encoder categories:
	[array(['localhost:8080', 'localhost:9090'], dtype=object)]
Generated column names:
	['host_localhost:8080', 'host_localhost:9090']


The host column is transformed into two binary features - method_GET, method_POST, method_PUT. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and concatenate the encoded features with the original DataFrame for modeling.

In [23]:
df_featurised[host_encoded_cols] = host_encoder.transform(df_featurised[category_col]).copy()
df_featurised.head(5)

,index,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090
0,6883,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,0.0
1,4687,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,0.0
2,25623,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,0.0
3,10367,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,0.0
4,19522,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0


### Analyse URL column values

In [31]:
print(f"Unique URLs: {df_featurised['url'].nunique()}")
# Random url samples
random_row_numbers = random.sample(range(0, 61066), 10)
for rn in random_row_numbers:
    print(f"row#{rn}: {df_featurised.iloc[rn]['url']}")

Unique URLs: 1643
row#13789: http://localhost:8080/tienda1/publico/autenticar.jsp
row#50530: http://localhost:8080/tienda1/publico/entrar.jsp/4861362529278789730
row#36668: http://localhost:8080/tienda1/publico/vaciar.jsp
row#22446: http://localhost:8080/tienda1/publico/registro.jsp
row#42910: http://localhost:8080/tienda1/publico/registro.jsp
row#27971: http://localhost:8080/tienda1/miembros/editar.jsp
row#21426: http://localhost:8080/tienda1/miembros/index.jsp
row#21375: http://localhost:8080/tienda1/publico/pagar.jsp
row#31113: http://localhost:8080/tienda1/publico/productos.jsp
row#57371: http://localhost:8080/tienda1/global/menum.jsp


URLs contain patterns, structures, and metadata that can be leveraged for feature engineering.
- url length
- Count of special chars: `/` `.` `?` `&` etc
- Presence of query prarams
- Presence of suspecious patterns: sql select, union, drop etc
- Presence of path traversal

In [70]:
from collections import defaultdict

#url_df = df_featurised["url"].head(10).to_frame()
url_df = df_featurised["url"].to_frame()

# Special characters
special_chars = ['/', '.', '?', '&', '=', '%']
total_urls_per_special_char = defaultdict(int)

# SQL patterns
sql_patterns = [ "select", "union", "drop", "insert", "delete"]
total_urls_per_sql_pattern = defaultdict(int)
# Initialize values to 0
for pattern in sql_patterns:
    total_urls_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_urls_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_urls_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_urls_with_or_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_urls_with_comment_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_urls_with_path_traversal_pattern = 0

# Parameter pattern
parameter_regex_pattern = r"[?&][^=&]+="
total_urls_with_parameter_pattern = 0

# Trailing digits
trailing_digits_regex_pattern= r"http://localhost:8080[^0-9]*(\d+)"
total_urls_with_trailing_digits_pattern = 0

# Numeric ID
numeric_id_regex_pattern = r"/\d{10,}"
total_urls_with_numeric_id_pattern = 0

# Iterate through each URL
for url in url_df["url"]:
    # Initialize a dictionary to store counts for each special character in this URL
    char_count = defaultdict(int)

    # Count occurrences of each special character in the URL
    for char in special_chars:
        count = url.count(char)
        char_count[char] = count

        # If the character appears at least once, increment the total URLs count
        if count > 0:
            total_urls_per_special_char[char] += 1
        else:
            total_urls_per_special_char[char] += 0

    for pattern in sql_patterns:
        if pattern in url:
          total_urls_per_sql_pattern[pattern] +=  1
        
    if re.search(script_regex_pattern, url):
        total_urls_with_script_pattern += 1
  
    if re.search(alert_regex_pattern, url):
        total_urls_with_alert_pattern += 1

    if re.search(or_regex_pattern, url):
        total_urls_with_or_pattern += 1
        
    if re.search(comment_regex_pattern, url):
        total_urls_with_comment_pattern += 1
        
    if re.search(path_traversal_regex_pattern, url):
        total_urls_with_path_traversal_pattern += 1

    if re.search(parameter_regex_pattern, url):
        total_urls_with_parameter_pattern += 1
    
    if re.search(trailing_digits_regex_pattern, url):
        total_urls_with_trailing_digits_pattern += 1

    if re.search(numeric_id_regex_pattern, url):
        total_urls_with_numeric_id_pattern += 1
    
# Display the total count of URLs containing each special character
print("Total URLs containing per special Character:")
for char, count in total_urls_per_special_char.items():
    print(f"    {char}: {count} URLs")

print("Total URLs containing per SQL pattern:")
for pattern, count in total_urls_per_sql_pattern.items():
    print(f"    {pattern}: {count} URLs")

print(f"Total URLs containing other patterns:")
print(f"   'or': {total_urls_with_or_pattern}")
print(f"   'script': {total_urls_with_script_pattern}")
print(f"   'alert': {total_urls_with_alert_pattern}")
print(f"   'comment': {total_urls_with_comment_pattern}")
print(f"   'path traversal': {total_urls_with_path_traversal_pattern}")
print(f"   'parameter': {total_urls_with_parameter_pattern}")
print(f"   'trailing digits': {total_urls_with_trailing_digits_pattern}")
print(f"   'numeric id': {total_urls_with_numeric_id_pattern}")


Total URLs containing per special Character:
    /: 61065 URLs
    .: 60386 URLs
    ?: 0 URLs
    &: 0 URLs
    =: 0 URLs
    %: 0 URLs
Total URLs containing per SQL pattern:
    select: 8 URLs
    union: 0 URLs
    drop: 5 URLs
    insert: 0 URLs
    delete: 0 URLs
Total URLs containing other patterns:
   'or': 0
   'script': 0
   'alert': 0
   'comment': 0
   'path traversal': 0
   'parameter': 0
   'trailing digits': 59329
   'numeric id': 967


**Observations:**

- Special Characters:  
  - `/` and `.` are too common and do not help distinguish anomalies.  
  - `?`, `&`, `=`, `%` do not appear in the dataset, so they provide no signal.  
  > **Conclusion:** Exclude all special character counts from the model

- SQL Patterns:  
  - `select` and `drop` are rare but meaningful for detecting SQL injection.  
  - `union`, `insert`, `delete` do not appear in the dataset.
  > **Conclusion:** Include `select` and `drop` as binary features to for detecting SQL injection attempts.

- Other Patterns:  
  - All patterns `or`, `script`, `alert`, `comment`, `path traversal`, `parameter` have zero occurrences in the dataset.  
  - `Trailing digits` appear in almost all URLs and are not discriminative.
  - `Numeric ID` appear in considerable URLs
  > **Conclusion:** Exclude all other patterns from the model.

In [60]:
import re
print('GeeksforGeeks:', re.search(r'\bGeeks\b', 'Geeks.forGeeks'))
if re.search(r'\bGeeks\b', 'Geeks.forGeeks'):
    print("matched")

GeeksforGeeks: <re.Match object; span=(0, 5), match='Geeks'>
matched


### Analyse `label` column values

In [ ]:
df_featurised['label'].value_counts()

In [ ]:
`label` is what type of feature category column ? give justificaiton for documentation

### Analyse `payload` column values

In [ ]:
print(f"Unique Payloads: {df_featurised['payload'].nunique()}")
unique_values = df_featurised['payload'].unique()
#print(type(unique_values))
#print(len(unique_values))
#df_featurised['url'].value_counts()
print("Few random sampled values:\n") 
random_row_numbers = random.sample(range(0, len(unique_values)), 10)
for rn in random_row_numbers:
    print(f"    {unique_values[rn]}\n")

In [ ]:
`payload` is what type of feature category column ? give justificaiton for documentation

### Analyse `contentLength` column values

In [ ]:
print(f"Unique values: {df_featurised['contentLength'].nunique()}")
df_featurised['contentLength'].describe()

`contentLength` is what type of feature category column ? give justificaiton for documentation

In [ ]:
# Add binary flag to indicate whether a request has a payload:
df["payload_is_empty"] = (payload_series == "").astype(int)

In [ ]:
df.info()

## 3. Encode Categorical Variables

Convert categorical columns `method`, `label`, `host` into a format suitable for machine learning (e.g., one-hot encoding, label encoding)

## 4. Create Domain-Specific Features  

Extract features that capture HTTP request characteristics (e.g., SQL injection patterns, payload length, URL length).


## 5. Save the Feature-Engineered DataFrame to disk

Save the final DataFrame (with features) as `csic_2010_engineered.csv` for modeling.